# Прогноз остатков с ARIMA и SARIMAX

Ноутбук сравнивает две модели на одном ряду `OST_VAL` с датами `REP_DT`. Для каждой модели получаем прогнозы на 1–12 месяцев. Дневные ряды остаются дневными, а ряды с одной строкой в месяц приводятся к концу месяца.

Если пакетов ещё нет, выполни в отдельной ячейке `%pip install statsmodels "mlflow>=3,<4" "pandas>=2.2" matplotlib`.

## 1. Данные и настройки

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import mlflow.statsmodels
import numpy as np
import pandas as pd
import statsmodels
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Загрузи свои данные в df или используй файл проекта с CURRENCY = "RUB".
# df = pd.read_csv("df_fl_synthetic.csv").drop_duplicates(["REP_DT", "CURENCY"])

CURRENCY = None        # Если валют несколько, укажи нужную, например "RUB".
HORIZONS = range(1, 13)
N_FOLDS = 3
TEST_START = None      # Или дата первого выпуска test, например "2024-01-01".
ORDER = (1, 1, 0)
SEASONAL_ORDER = None  # По умолчанию недельная сезонность для дневного ряда.
FUTURE_COLS = []       # Числовые признаки, известные заранее на весь горизонт.
STAT_COLS = []         # Числовые статистики, доступные на дату строки.

In [ ]:
data = df.copy()
currency_col = next((col for col in ("CURENCY", "CURRENCY") if col in data), None)
if currency_col:
    data = data.dropna(subset=[currency_col])
    if CURRENCY is None and data[currency_col].nunique() > 1:
        raise ValueError("Укажи CURRENCY в ячейке настроек.")
    if CURRENCY is not None:
        data = data[data[currency_col] == CURRENCY].copy()

dates = pd.to_datetime(data["REP_DT"], format="mixed", errors="coerce")
data["REP_DT"] = dates.dt.tz_localize(None).dt.normalize()
data = data.dropna(subset=["REP_DT"]).set_index("REP_DT").sort_index()
if data.empty or data.index.has_duplicates:
    raise ValueError("Нужен один ряд остатков с одной строкой на дату.")

monthly = data.index.to_period("M").is_unique
if monthly:
    data.index = data.index + pd.offsets.MonthEnd(0)
data = data.asfreq("ME" if monthly else "D")
y = pd.to_numeric(data["OST_VAL"], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
season = SEASONAL_ORDER or ((0, 0, 0, 0) if monthly else (1, 0, 0, 7))

## 2. Признаки SARIMAX

По умолчанию используются календарные признаки. Пользовательские `FUTURE_COLS` должны быть известны на весь путь прогноза и заполнены для каждой даты. Колонки из `STAT_COLS` сдвигаются на 12 месяцев для месячного ряда или на 366 дней для дневного. Так прогноз на любой из 12 горизонтов использует только уже доступные статистики.

In [ ]:
exog = pd.DataFrame(index=y.index)
if monthly:
    angle = 2 * np.pi * y.index.month / 12
    exog["calendar_month_sin"] = np.sin(angle)
    exog["calendar_month_cos"] = np.cos(angle)
else:
    exog["calendar_weekend"] = (y.index.dayofweek >= 5).astype(float)
    exog["calendar_month_end"] = y.index.is_month_end.astype(float)

if "OST_VAL" in FUTURE_COLS or set(FUTURE_COLS) & set(STAT_COLS):
    raise ValueError("Укажи разные наборы признаков. Остаток можно добавить в STAT_COLS с лагом.")
future = data[FUTURE_COLS].apply(pd.to_numeric, errors="coerce").astype(float)
if not np.isfinite(future.to_numpy()).all():
    raise ValueError("Заполни FUTURE_COLS числовыми значениями на всей календарной сетке.")
stats = data[STAT_COLS].apply(pd.to_numeric, errors="coerce").astype(float)
stats = stats.replace([np.inf, -np.inf], np.nan).ffill().shift(12 if monthly else 366)
exog = exog.join(future).join(stats).dropna()
y = y.reindex(exog.index).asfreq("ME" if monthly else "D")
exog = exog.reindex(y.index)

## 3. Даты выпуска и временные фолды

Прогноз оценивается на общей сетке выпусков для обоих методов. В дневном ряду берём каждую седьмую подходящую дату выпуска, в месячном — каждую. Между выпусками модели получают уже наблюдавшиеся остатки. Для сравнения горизонтов берём выпуски с известным фактом на максимальном горизонте.

Факты из `valid` должны быть известны к началу теста. Каждый фолд оценивает свой блок дат выпуска. Для `train` используются параметры первого фолда и прогнозы внутри его обучающего периода.

In [ ]:
HORIZONS = sorted(set(HORIZONS))
if not HORIZONS or any(not isinstance(h, (int, np.integer)) or not 1 <= h <= 12 for h in HORIZONS):
    raise ValueError("HORIZONS должен содержать целое число месяцев от 1 до 12.")

target_positions = np.column_stack([
    y.index.get_indexer(y.index + (pd.offsets.MonthEnd(h) if monthly else pd.DateOffset(months=h)))
    for h in HORIZONS
])
observed = y.notna().to_numpy()
last_target = target_positions[:, -1]
origins = np.flatnonzero(observed & (last_target >= 0) & observed[last_target])
if len(origins) < 3:
    raise ValueError("Мало истории для максимального горизонта.")

default_test = y.index[origins[int(0.8 * len(origins))]]
test_date = pd.Timestamp(TEST_START or default_test).tz_localize(None).normalize()
test_start = y.index.searchsorted(test_date)
# Оставляем несколько выпусков для оценки train на максимальном горизонте.
validation = origins[(origins > last_target[origins[2]]) & (last_target[origins] < test_start)]
validation = validation[len(validation) // 2:]
test_origins = origins[origins >= test_start]
if len(validation) < N_FOLDS or not len(test_origins):
    raise ValueError(f"Мало истории для {N_FOLDS} фолдов и теста. Проверь TEST_START и диапазон дат.")

step = 1 if monthly else 7
folds = [block[::step] for block in np.array_split(validation, N_FOLDS)]
train_origins = origins[(origins > 0) & (last_target[origins] < folds[0][0])][::step]
test_origins = test_origins[::step]
print(f"Начало test {test_date.date()}, выпусков test {len(test_origins)}")

## 4. Обучение и прогнозы

На каждом фолде оцениваем параметры на истории до начала блока. Затем обновляем состояние модели фактами по мере их появления. Перед тестом один раз переобучаем параметры. Из каждого прогноза берём значения на нужных календарных горизонтах.

In [ ]:
def fit(name, stop):
    train = y.iloc[:stop]
    center = train.mean()
    scale = train.std(ddof=0) or 1.0
    z = (y - center) / scale
    x = None if name == "ARIMA" else exog
    if name == "ARIMA":
        result = ARIMA(z.iloc[:stop], order=ORDER).fit()
    else:
        result = SARIMAX(
            z.iloc[:stop], exog=x.iloc[:stop], order=ORDER, seasonal_order=season
        ).fit(disp=False)
    return result, z, x, center, scale

def forecast(fitted, origins, split, fold=None):
    result, z, x, center, scale = fitted
    seen = result.nobs
    parts = []
    for origin in origins:
        origin = int(origin)
        targets = target_positions[origin]
        end = int(targets[-1])
        if split == "train":
            pred = result.get_prediction(start=origin + 1, end=end, dynamic=True)
        else:
            history_x = x.iloc[seen:origin + 1] if x is not None else None
            result = result.extend(z.iloc[seen:origin + 1], exog=history_x)
            seen = origin + 1
            future_x = x.iloc[seen:end + 1] if x is not None else None
            pred = result.get_forecast(steps=end - origin, exog=future_x)

        offsets = targets - origin - 1
        lower, upper = (pred.conf_int().to_numpy()[offsets] * scale + center).T
        parts.append(pd.DataFrame({
            "REP_DT": y.index[origin], "target_dt": y.index[targets], "months": list(HORIZONS),
            "target": y.iloc[targets].to_numpy(), "baseline": y.iloc[origin],
            "forecast": pred.predicted_mean.to_numpy()[offsets] * scale + center,
            "lower": lower, "upper": upper, "split": split, "fold": fold,
        }))
    return pd.concat(parts, ignore_index=True).dropna(subset=["target"])

def score(rows, by):
    error = (rows["target"] - rows["forecast"]).abs()
    baseline = (rows["target"] - rows["baseline"]).abs()
    denom = rows["target"].abs().replace(0, np.nan)
    return rows.assign(
        MAE=error, MAPE_pct=100 * error / denom,
        baseline_MAE=baseline, baseline_MAPE_pct=100 * baseline / denom,
    ).groupby(by, as_index=False).agg(
        MAE=("MAE", "mean"), MAPE_pct=("MAPE_pct", "mean"),
        baseline_MAE=("baseline_MAE", "mean"), baseline_MAPE_pct=("baseline_MAPE_pct", "mean"),
        n=("target", "size"), n_MAPE=("MAPE_pct", "count"),
    )

In [ ]:
if not mlflow.config.is_tracking_uri_set():
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("arima-sarimax-balances")
client = mlflow.MlflowClient()

models, run_ids, model_uris, target_scaling = {}, {}, {}, {}
predictions, metrics, cv_metrics = [], [], []
for name in ("ARIMA", "SARIMAX"):
    parts = []
    for fold, dates in enumerate(folds, start=1):
        fitted = fit(name, dates[0])
        if fold == 1:
            parts.append(forecast(fitted, train_origins, "train", fold))
        parts.append(forecast(fitted, dates, "valid", fold))

    fitted = fit(name, test_start)
    model, _, _, center, scale = fitted
    models[name] = model
    target_scaling[name] = (center, scale)
    parts.append(forecast(fitted, test_origins, "test"))
    rows = pd.concat(parts, ignore_index=True).assign(model=name)
    if not np.isfinite(rows["forecast"]).all():
        raise ValueError(f"{name} вернула некорректный прогноз. Проверь порядок модели и признаки.")

    scores = score(rows, ["model", "months", "split"])
    fold_scores = score(rows[rows["split"] == "valid"], ["model", "months", "fold"])
    predictions.append(rows)
    metrics.append(scores)
    cv_metrics.append(fold_scores)

    with mlflow.start_run(run_name=name) as run:
        run_ids[name] = run.info.run_id
        mlflow.log_params({
            "order": ORDER, "seasonal_order": season if name == "SARIMAX" else None,
            "test_start": str(test_date.date()), "n_folds": N_FOLDS,
            "target_center": center, "target_scale": scale,
            "exog": exog.columns.tolist() if name == "SARIMAX" else [],
            "converged": model.mle_retvals.get("converged"),
        })
        for row in scores.to_dict("records"):
            prefix = f"{row['split']}/{row['months']}m"
            mlflow.log_metrics({
                f"{prefix}/{key}": row[key] for key in ("MAE", "MAPE_pct") if pd.notna(row[key])
            })
        mlflow.log_text(scores.to_csv(index=False), "metrics.csv")
        mlflow.log_text(fold_scores.to_csv(index=False), "cv_metrics.csv")
        model_info = mlflow.statsmodels.log_model(
            model, name=name.lower(),
            pip_requirements=[f"statsmodels=={statsmodels.__version__}", f"pandas=={pd.__version__}"],
        )
        model_uris[name] = model_info.model_uri

predictions = pd.concat(predictions, ignore_index=True)
metrics = pd.concat(metrics, ignore_index=True)
cv_metrics = pd.concat(cv_metrics, ignore_index=True)

## 5. Качество на тесте и по фолдам

Базовый прогноз равен остатку на дату выпуска. MAPE считается по ненулевым фактам. В локальном MLflow открой результаты командой `mlflow ui --backend-store-uri sqlite:///mlflow.db`. При загрузке модели из MLflow итоговый прогноз получается как `raw_forecast * target_scale + target_center`.

In [ ]:
display(metrics.query("split == 'test'").sort_values(["months", "model"]))
display(cv_metrics)

## 6. Факт и прогноз

In [ ]:
PLOT_MONTHS = 3 if 3 in HORIZONS else HORIZONS[0]
for name in models:
    rows = predictions.query("model == @name and months == @PLOT_MONTHS").sort_values("target_dt")
    fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
    ax.plot(y.index, y, color="0.45", linewidth=1, label="Факт")
    for split, part in rows.groupby("split", sort=False):
        ax.plot(part["target_dt"], part["forecast"], marker=".", markersize=3, label=split)
        if split == "test":
            ax.fill_between(part["target_dt"], part["lower"], part["upper"],
                            color="C2", alpha=0.15, label="95% интервал прогноза test")
    ax.set(title=f"{name} · {PLOT_MONTHS} мес.", xlabel="Дата целевого остатка", ylabel="OST_VAL")
    ax.legend()
    ax.grid(alpha=0.2)
    client.log_figure(run_ids[name], fig, f"forecast_{PLOT_MONTHS}m.png")
    plt.show()

## 7. Коэффициенты моделей

Для ARIMA смотрим коэффициенты лагов и ошибок. Для SARIMAX дополнительно строим график коэффициентов внешних признаков с 95% доверительными интервалами. На графике коэффициент показывает изменение в единицах `OST_VAL` на единицу признака при фиксированных остальных компонентах модели.

In [ ]:
tables = []
for name, model in models.items():
    table = model.params.rename("estimate").to_frame()
    table[["lower", "upper"]] = model.conf_int().to_numpy()
    table = table.rename_axis("term").reset_index().assign(model=name)
    tables.append(table)
    client.log_text(run_ids[name], table.to_csv(index=False), "coefficients.csv")

coefficients = pd.concat(tables, ignore_index=True)
display(coefficients)
sarimax_exog = (coefficients["model"] == "SARIMAX") & coefficients["term"].isin(exog.columns)
effects = coefficients[sarimax_exog].set_index("term")[["estimate", "lower", "upper"]]
effects = effects * target_scaling["SARIMAX"][1]
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.errorbar(effects["estimate"], effects.index, fmt="o",
            xerr=[effects["estimate"] - effects["lower"], effects["upper"] - effects["estimate"]])
ax.axvline(0, color="0.5", linewidth=1)
ax.set(title="Коэффициенты признаков SARIMAX", xlabel="Изменение OST_VAL на единицу признака")
client.log_figure(run_ids["SARIMAX"], fig, "coefficients.png")
plt.show()